<a href="https://colab.research.google.com/github/Nick97382000/Lettura_bilanci/blob/main/Lettura%20bilanci.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Lettura bilanci

Installo librerie di interesse e copio directory di riferimento da Git-Hub

In [7]:
!git clone https://github.com/Nick97382000/Lettura_bilanci

import sys
sys.path.insert(0, '/content/Lettura_bilanci')

import importlib, dependencies
importlib.reload(dependencies)

Cloning into 'Lettura_bilanci'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 189 (delta 16), reused 0 (delta 0), pack-reused 156 (from 1)
Receiving objects: 100% (189/189), 29.91 MiB | 33.22 MiB/s, done.
Resolving deltas: 100% (91/91), done.
Installing Python package: pdfplumber...
Installing Python package: pdf2image...
Installing Python package: pytesseract...
Installing Python package: pillow...
Updating apt-get...
Installing apt packages: tesseract-ocr, tesseract-ocr-ita, poppler-utils...
Installing Python package: pillow...
Updating apt-get...
Installing apt packages: tesseract-ocr, tesseract-ocr-ita, poppler-utils...


<module 'dependencies' from '/content/Lettura_bilanci/dependencies.py'>

In [29]:
!pip install PyMuPDF

Aggiorno prendendo le modifiche in git hub

In [9]:
!git -C /content/Lettura_bilanci pull   # aggiorna i file

Already up to date.


Carico packages di interesse

In [10]:
import formato_oic.config as cf, formato_oic.ocr_func as ocr, formato_oic.analizza_oic as oic_an
importlib.reload(cf)
importlib.reload(ocr)
importlib.reload(oic_an)

<module 'formato_oic.analizza_oic' from '/content/Lettura_bilanci/formato_oic/analizza_oic.py'>

Leggo tutti i pdf nella cartella e stampo la tabella di sintesi

In [11]:
import glob
import os
import pandas as pd

pdf_files = glob.glob("Lettura_bilanci/deposito bilanci/*.pdf")

summary = {}

for pdf_path in pdf_files:
    nome = os.path.basename(pdf_path).replace(".pdf", "")
    tabella = oic_an.trova_tabella_bilancio(pdf_path, cf.keywords_first_page)

    if tabella is not None:
        summary[nome] = oic_an.summary_tabella(tabella, cf.campi_bilancio)



# stampa i risultati ottenuti
for nome, df in summary.items():
    print("\n" + "=" * 80)
    print(f"BILANCIO: {nome}")
    print("=" * 80)
    print(df)


BILANCIO: Valtellina - 2024
                                            31-12-2024   31-12-2023
totale_attivo                              250006305.0  229638858.0
patrimonio_netto                            61305455.0   56175195.0
totale_attivo_circolante                   187543091.0  185084246.0
Totale_disponibilità_liquide                10973940.0   16503906.0
Totale_debiti                              162338791.0  147569430.0
Totale_rimanenze                           110566346.0   91968335.0
Totale_crediti_verso_clienti                45713002.0   50821290.0
Totale_debiti_verso_fornitori               93300761.0   94223386.0
Totale_debiti_verso_banche                  40085441.0   24659657.0
avviamento                                     20000.0      87000.0
ricavi_vendite_e_prestazioni               290308450.0  280124262.0
Utile_perdita_esercizio                      5103126.0    4215880.0
Risultato_prima_delle_imposte                8922840.0    8670937.0
Totale_ammortamenti

Debug

In [12]:

pdf_path= "Lettura_bilanci/deposito bilanci/Conserve Italia - Civilistico e Consolidato - 2024.pdf"
#pdf_path= "Lettura_bilanci/deposito bilanci/Gollinucci - 2024.pdf"
tabella = oic_an.trova_tabella_bilancio(pdf_path, cf.keywords_first_page)
oic_an.summary_tabella(tabella, cf.campi_bilancio)


,30 giugno 2025,30 giugno 2024
totale_attivo,932834011.0,879915161.0
patrimonio_netto,305929558.0,300611685.0
totale_attivo_circolante,439099320.0,416207934.0
Totale_disponibilità_liquide,127609819.0,106847278.0
Totale_debiti,591475140.0,545013151.0
Totale_rimanenze,205612371.0,201996264.0
Totale_crediti_verso_clienti,55501954.0,60932295.0
Totale_debiti_verso_fornitori,236897353.0,228044458.0
Totale_debiti_verso_banche,191590518.0,168282068.0
avviamento,NaN,NaN


In [13]:
"""
df = ocr.trova_tabella_oic_multipaagina(
    pdf_path="Lettura_bilanci/deposito bilanci/Kirey - 2024.pdf",
    start_marker="stato patrimoniale",
    end_marker="conto economico",
    lang="ita",
    dpi=300
)

df.head(50)
df.tail(50)
"""

'\ndf = ocr.trova_tabella_oic_multipaagina(\n    pdf_path="Lettura_bilanci/deposito bilanci/Kirey - 2024.pdf",\n    start_marker="stato patrimoniale",\n    end_marker="conto economico",\n    lang="ita",\n    dpi=300\n)\n\ndf.head(50)\ndf.tail(50)\n'

In [14]:
"""
print(df)
"""

'\nprint(df)\n'

In [30]:
import fitz # PyMuPDF
import pandas as pd

def extract_text_with_styles_pymupdf(pdf_path):
    """
    Extracts text along with its font size and font name from a PDF using PyMuPDF.
    Each span (a segment of text with consistent style) is treated as a text element.
    """
    all_text_info = []
    try:
        doc = fitz.open(pdf_path)
        for page_num, page in enumerate(doc):
            # get_text('dict') provides a structured output of blocks, lines, and spans
            # Removed `flags` argument due to compatibility issue.
            text_data = page.get_text("dict")
            for block in text_data['blocks']:
                if block['type'] == 0: # text block
                    for line in block['lines']:
                        for span in line['spans']:
                            text = span['text'].strip()
                            if text: # Only process non-empty text spans
                                fontname = span['font']
                                size = span['size']
                                x0, y0, x1, y1 = span['bbox']
                                all_text_info.append({
                                    'page': page_num + 1,
                                    'text': text,
                                    'fontname': fontname,
                                    'size': size,
                                    'x0': x0, 'y0': y0, 'x1': x1, 'y1': y1
                                })
    except Exception as e:
        print(f"Error processing PDF {pdf_path}: {e}")
    finally:
        if 'doc' in locals() and doc.is_closed == False:
            doc.close()
    return pd.DataFrame(all_text_info)

# Example PDF path - using Gollinucci - 2024.pdf
sample_pdf_path = "Lettura_bilanci/deposito bilanci/IFB SRL - 2024.pdf"

# Extract text with style information using PyMuPDF
text_df = extract_text_with_styles_pymupdf(sample_pdf_path)

print(f"Extracted {len(text_df)} text elements from {sample_pdf_path} using PyMuPDF")
display(text_df.head())


Extracted 1571 text elements from Lettura_bilanci/deposito bilanci/IFB SRL - 2024.pdf using PyMuPDF


,page,text,fontname,size,x0,y0,x1,y1
0,1,I.F.B. S.R.L.,Helvetica-Bold,18.0,28.346001,176.049026,128.371994,200.835022
1,1,estratto il 18/09/2025 alle ore 16:07:30,Helvetica,12.0,28.346001,197.451019,233.137939,213.939011
2,1,Documento Richiesto,Helvetica-Bold,10.0,36.686001,273.360962,139.475998,287.130981
3,1,DIB Documenti ed Informazioni relative al Bila...,Helvetica-Bold,10.0,36.686001,292.485962,396.825897,306.255981
4,1,Denominazione:,Helvetica,10.0,80.875000,312.435974,153.124985,326.175964


I've replaced the `pdfplumber` based extraction with `PyMuPDF` (imported as `fitz`). The `extract_text_with_styles_pymupdf` function now processes the PDF page by page, and for each page, it extracts text in 'spans'. A 'span' in `PyMuPDF` refers to a contiguous sequence of characters that share the same font, size, and color. This unit is well-suited for capturing both the text and its associated style information.

The subsequent clustering logic based on font size should still work with this new `text_df` format, as the `fontname` and `size` columns are preserved.

In [16]:
display(text_df[90:100])

,page,text,fontname,size,x0,y0,x1,y1
90,3,2.538.194,Helvetica,8.0,526.341980,178.343964,561.925964,189.335968
91,3,Utile/Perdita,Helvetica,8.0,33.346001,194.511017,76.914001,205.503021
92,3,1.786.656,Helvetica,8.0,280.674011,193.093964,316.257996,204.085968
93,3,2.172.984,Helvetica,8.0,403.507996,193.093964,439.091980,204.085968
94,3,642.760,Helvetica,8.0,533.013977,193.093964,561.925964,204.085968
95,3,- +A riserve/-Distr. riserve,Helvetica-Oblique,8.0,37.793999,209.301025,128.690002,220.133026
96,3,1.786.656,Helvetica,8.0,280.674011,207.843964,316.257996,218.835968
97,3,2.172.984,Helvetica,8.0,403.507996,207.843964,439.091980,218.835968
98,3,642.760,Helvetica,8.0,533.013977,207.843964,561.925964,218.835968
99,3,- Altre distribuzioni,Helvetica-Oblique,8.0,37.793999,224.051025,103.585999,234.883026


Now that we have extracted the text along with its font and size, we need to identify potential paragraph titles. A common heuristic is that titles tend to have larger font sizes than the main body text. We will analyze the distribution of font sizes to determine a threshold for titles.

After identifying titles, we will group the text into sections.

In [28]:
import numpy as np
import pandas as pd # Ensure pandas is imported as pd for Series
import re # Import re for regex operations

def clean_page_numbers_from_title(title_text):
    """
    Removes 'pag. N di N' or similar patterns from a given title string.
    """
    # Regex to find 'pag. N di N' or 'page N of N' patterns
    # Handles variations in spacing and case, and both 'di' and 'of'
    pattern = r'\b(pag\.?|page)\s*\d+\s*(di|of)\s*\d+\b'
    cleaned_title = re.sub(pattern, '', title_text, flags=re.IGNORECASE).strip()
    # Remove any extra spaces that might result from the substitution
    cleaned_title = re.sub(r'\s+', ' ', cleaned_title).strip()
    return cleaned_title

def identify_main_sections_meta(text_df, title_size_threshold_multiplier=1.6, positional_title_size_threshold_multiplier=1.2, y0_page_top_threshold=100):
    """
    Identifies potential main titles based on font size and position, and returns their metadata.
    Titles before the last 'nota' title must contain 'nota'.
    After the last 'nota' title, any sufficiently large title is considered valid.
    This function now returns a list of dictionaries, each containing 'title', 'start_idx', and 'end_idx'.
    Content before the first 'nota' title is explicitly discarded.
    """
    if text_df.empty:
        return []

    # Calculate the median font size for body text (excluding very large/small outliers initially)
    size_counts = text_df['size'].value_counts()
    common_sizes = size_counts[size_counts > 5].index.tolist()

    if not common_sizes:
        median_body_font_size = text_df['size'].median()
    else:
        median_body_font_size = pd.Series(common_sizes).median()

    primary_title_size_threshold = median_body_font_size * title_size_threshold_multiplier
    positional_title_size_threshold = median_body_font_size * positional_title_size_threshold_multiplier

    print(f"Median body font size: {median_body_font_size:.2f}")
    print(f"Primary Title font size threshold (>{primary_title_size_threshold:.2f}):")
    print(f"Positional Title font size threshold (>{positional_title_size_threshold:.2f} AND y0 < {y0_page_top_threshold}):")

    # --- Pre-scan to find the start index of the last valid 'nota' title ---
    last_nota_title_start_idx = -1
    temp_buffer_for_scan = []
    potential_title_sequence_first_idx_scan = -1
    previous_page_for_scan = -1

    for j, scan_row in text_df.iterrows():
        is_primary_title_font_scan = scan_row['size'] > primary_title_size_threshold
        is_new_page_start_scan = (scan_row['page'] != previous_page_for_scan)
        is_at_top_of_page_scan = (scan_row['y0'] < y0_page_top_threshold)
        is_positional_title_font_scan = (scan_row['size'] > positional_title_size_threshold)
        is_scan_word_title_candidate = is_primary_title_font_scan or (is_new_page_start_scan and is_at_top_of_page_scan and is_positional_title_font_scan)
        previous_page_for_scan = scan_row['page']

        if is_scan_word_title_candidate:
            if not temp_buffer_for_scan: # If this is the first word of a potential title sequence
                potential_title_sequence_first_idx_scan = j
            temp_buffer_for_scan.append(scan_row['text'])
        else:
            if temp_buffer_for_scan: # Evaluate the sequence that just ended
                potential_title_text_scan = " ".join(temp_buffer_for_scan).strip()
                cleaned_title_scan = clean_page_numbers_from_title(potential_title_text_scan)
                if cleaned_title_scan and "nota" in cleaned_title_scan.lower():
                    last_nota_title_start_idx = potential_title_sequence_first_idx_scan
                temp_buffer_for_scan = []
                potential_title_sequence_first_idx_scan = -1

    if temp_buffer_for_scan and potential_title_sequence_first_idx_scan != -1:
        potential_title_text_scan = " ".join(temp_buffer_for_scan).strip()
        cleaned_title_scan = clean_page_numbers_from_title(potential_title_text_scan)
        if cleaned_title_scan and "nota" in cleaned_title_scan.lower():
            last_nota_title_start_idx = potential_title_sequence_first_idx_scan

    print(f"Last 'nota' title starts at text_df index: {last_nota_title_start_idx}")
    # --- End of pre-scan ---


    sections_meta = []
    current_main_title = None
    current_main_section_start_idx = -1
    has_found_first_nota_title = False

    temp_title_buffer = []
    potential_title_sequence_first_idx = -1
    previous_page = -1

    for i, row in text_df.iterrows():
        is_primary_title_font = row['size'] > primary_title_size_threshold
        is_new_page_start = (row['page'] != previous_page)
        is_at_top_of_page = (row['y0'] < y0_page_top_threshold)
        is_positional_title_font = (row['size'] > positional_title_size_threshold)
        is_current_word_title_candidate = is_primary_title_font or (is_new_page_start and is_at_top_of_page and is_positional_title_font)
        previous_page = row['page']

        if is_current_word_title_candidate:
            if not temp_title_buffer:
                potential_title_sequence_first_idx = i
            temp_title_buffer.append(row['text'])
        else:
            if temp_title_buffer: # A sequence of high-font words just ended. Evaluate it.
                potential_title_text = " ".join(temp_title_buffer).strip()
                cleaned_title = clean_page_numbers_from_title(potential_title_text)

                is_after_last_nota_title_point = (last_nota_title_start_idx != -1 and potential_title_sequence_first_idx >= last_nota_title_start_idx)

                is_valid_title_this_pass = False
                if is_after_last_nota_title_point:
                    is_valid_title_this_pass = bool(cleaned_title)
                else:
                    is_valid_title_this_pass = bool(cleaned_title) and ("nota" in cleaned_title.lower())

                if is_valid_title_this_pass:
                    if not has_found_first_nota_title: # This is the first valid 'nota' title encountered
                        has_found_first_nota_title = True
                        # Discard any content accumulated before this point

                    if current_main_title is not None: # If there was a previous active section
                        sections_meta.append({
                            'title': current_main_title,
                            'start_idx': current_main_section_start_idx,
                            'end_idx': potential_title_sequence_first_idx -1 # End previous section just before this new title
                        })

                    current_main_title = cleaned_title
                    current_main_section_start_idx = potential_title_sequence_first_idx

                temp_title_buffer = []
                potential_title_sequence_first_idx = -1

    # After loop, handle the last section if any is active and a 'nota' title was found
    if current_main_title is not None and has_found_first_nota_title:
        sections_meta.append({
            'title': current_main_title,
            'start_idx': current_main_section_start_idx,
            'end_idx': len(text_df) - 1
        })

    return sections_meta, median_body_font_size

def cluster_subsections(section_df_slice, median_body_font_size, subtitle_size_multiplier=1.2):
    """
    Identifies sub-titles within a given section's DataFrame slice and clusters its content.
    Returns a list of dictionaries, each with 'title' and 'content_words'.
    """
    if section_df_slice.empty:
        return []

    raw_subsections = []
    current_subtitle = "Section Introduction" # Initial "title" for content before first recognized subtitle
    current_subsection_content_words = []

    # Subtitle threshold is lower than main title threshold but still higher than body text
    subtitle_size_threshold = median_body_font_size * subtitle_size_multiplier
    print(f"  Subtitle font size threshold (>{subtitle_size_threshold:.2f})")

    temp_subtitle_buffer = []

    for i, row in section_df_slice.iterrows():
        is_subtitle_font = row['size'] > subtitle_size_threshold

        if is_subtitle_font:
            temp_subtitle_buffer.append(row['text'])
        else:
            if temp_subtitle_buffer:
                potential_subtitle_text = " ".join(temp_subtitle_buffer).strip()
                cleaned_subtitle = clean_page_numbers_from_title(potential_subtitle_text)

                if cleaned_subtitle: # Valid new subtitle
                    if current_subsection_content_words: # Save content for previous subsection
                        raw_subsections.append({
                            'title': current_subtitle,
                            'content_words': current_subsection_content_words
                        })

                    current_subtitle = cleaned_subtitle
                    current_subsection_content_words = []
                else: # False positive for subtitle, treat as regular content
                    current_subsection_content_words.extend(temp_subtitle_buffer)

                temp_subtitle_buffer = []

            current_subsection_content_words.append(row['text'])

    # After loop, handle any remaining content and subtitle buffer
    if temp_subtitle_buffer:
        potential_subtitle_text = " ".join(temp_subtitle_buffer).strip()
        cleaned_subtitle = clean_page_numbers_from_title(potential_subtitle_text)
        if cleaned_subtitle:
            if current_subsection_content_words:
                raw_subsections.append({
                    'title': current_subtitle,
                    'content_words': current_subsection_content_words
                })
            current_subtitle = cleaned_subtitle
            current_subsection_content_words = []
        else:
            current_subsection_content_words.extend(temp_subtitle_buffer)

    if current_subsection_content_words:
        raw_subsections.append({
            'title': current_subtitle,
            'content_words': current_subsection_content_words
        })

    return raw_subsections

def merge_short_subsections(raw_subsections_list, min_length=150):
    """
    Merges short subsections with preceding or succeeding ones based on content length.
    If the first subsection is short and there's a second one, it merges into the second.
    Otherwise, short subsections merge backward.
    """
    if not raw_subsections_list:
        return {}

    final_sections = []
    i = 0

    # Handle the very first section with a special rule if it's short and has a neighbor
    first_section = raw_subsections_list[0]
    first_content_str = " ".join(first_section['content_words'])

    if len(first_content_str) < min_length and len(raw_subsections_list) > 1:
        # First section is short and there's a second one, merge its content into the second's content
        # We will process the second section next, with this content prepended.
        raw_subsections_list[1]['content_words'] = first_section['content_words'] + raw_subsections_list[1]['content_words']
        i = 1 # Start processing from the second section
    else:
        # First section is long enough, or it's the only section (short or long), so add it directly
        final_sections.append(first_section)
        i = 1 # Start processing from the second section (if any)

    # Process remaining sections
    while i < len(raw_subsections_list):
        current_item = raw_subsections_list[i]
        current_content_str = " ".join(current_item['content_words'])

        if len(current_content_str) < min_length:
            if final_sections: # If there's a previous section to merge with
                final_sections[-1]['content_words'].extend(current_item['content_words'])
            else:
                # This path should ideally not be hit if the initial handling of `first_section` is correct.
                # If it is hit, it means a short section cannot merge backward or forward, so it stands alone.
                final_sections.append(current_item)
        else:
            final_sections.append(current_item)
        i += 1

    # Convert the list of dicts to the final dictionary format
    final_dict = {}
    for item in final_sections:
        final_dict[item['title']] = " ".join(item['content_words'])
    return final_dict

# Perform main section identification
sections_meta, median_body_font_size = identify_main_sections_meta(text_df,
                                                    title_size_threshold_multiplier=1.6,
                                                    positional_title_size_threshold_multiplier=1.2,
                                                    y0_page_top_threshold=100)

final_clustered_output = {}
print(f"\nDetected {len(sections_meta)} main sections:")
for meta in sections_meta:
    main_title = meta['title']
    start_idx = meta['start_idx']
    end_idx = meta['end_idx']

    # Slice the original text_df for this main section's content
    section_df_slice = text_df.loc[start_idx : end_idx].copy() # Use .copy() to avoid SettingWithCopyWarning

    # Perform sub-section clustering for this main section (returns list of dicts)
    raw_sub_sections_list = cluster_subsections(section_df_slice, median_body_font_size, subtitle_size_multiplier=1.2)
    # Merge short subsections and convert to final dict format
    merged_sub_sections = merge_short_subsections(raw_sub_sections_list, min_length=150) # Default min_length=150 characters
    final_clustered_output[main_title] = merged_sub_sections


# Display the final nested structure
for main_title, sub_sections in final_clustered_output.items():
    print("\n" + "=" * 50)
    print(f"MAIN SECTION: {main_title}")
    print("=" * 50)
    if sub_sections:
        for sub_title, content in sub_sections.items():
            print("  " + "-" * 40)
            print(f"  SUB-SECTION: {sub_title}")
            print(f"  Content Sample: {content[:150]}...")
            print("  " + "-" * 40)
    else:
        print("  No sub-sections detected, or section is empty.")


Median body font size: 10.01
Primary Title font size threshold (>16.02):
Positional Title font size threshold (>12.01 AND y0 < 100):
Last 'nota' title starts at text_df index: 1332

Detected 9 main sections:
  Subtitle font size threshold (>12.01)
  Subtitle font size threshold (>12.01)
  Subtitle font size threshold (>12.01)
  Subtitle font size threshold (>12.01)
  Subtitle font size threshold (>12.01)
  Subtitle font size threshold (>12.01)
  Subtitle font size threshold (>12.01)
  Subtitle font size threshold (>12.01)
  Subtitle font size threshold (>12.01)

MAIN SECTION: Nota integrativa al Bilancio di esercizio chiuso al 31-12-2024 Nota integrativa, parte iniziale
  ----------------------------------------
  SUB-SECTION: Nota integrativa al Bilancio di esercizio chiuso al 31-12-2024 Nota integrativa, parte iniziale
  Content Sample: Nota Integrativa al bilancio di esercizio al 31/12/2024 redatta in forma abbreviata ai sensi dell'art. 2435 bis del Codice Civile PREMESSA Il bilanci

The `identify_titles_and_cluster` function attempts to detect titles by finding text elements with a font size significantly larger than the median font size found in the document. It then groups the subsequent text until another title is encountered.

You can adjust the `title_size_threshold_multiplier` parameter in the `identify_titles_and_cluster` function to fine-tune what is considered a title. A higher multiplier will be stricter, only picking very large fonts as titles, while a lower one will be more lenient.

# I bilanci in Italia

## Tipi di bilanci in Italia
### A seconda dei principi contabili applicati (e del formato del bilancio)
- **Bilanci OIC (o IV Direttiva UE):**

  - sono pubblicati annualmente da tutte le Società di Capitali italiane (comprese le Società Cooperative)

  - il contenuto è normato dal Codice Civile (quindi: il formato è complessivamente stabile)

  - i principi contabili applicati sono quelli italiani (OIC significa Organismo Italiano Contabile)

- **Bilanci IAS/IFRS:**

  - sono pubblicati semestralmente dalle Società quotate, o annualmente da ogni Società di Capitali che decide di applicare i principi IAS/IFRS (abbastanza inconsueto)[^1].

  - il contenuto non è normato dalle Leggi italiane (ma fa riferimento ai documenti IAS/IFRS internazionali). In generale, la struttura è meno rigida di un bilancio OIC.

  - i principi contabili applicati sono quelli internazionali (IAS è l'acronimo di International Accounting Standards)

[^1]: Passare dai principi italiani a quelli internazionali ha un costo in termine di procedure, di formazione, di processi,
che non genera nessun ricavo. Questo fa sì che solo chi è obbligato per legge faccia di fatto bilanci IAS/IFRS.

### A seconda del perimetro considerato
Il bilancio è un documento che rappresenta le transazioni che avvengono tra una azienda, o un gruppo di aziende, e il mondo esterno.

Quindi hai un perimetro: solo le transazioni che attraversano il perimetro sono considerate significative. La contabilità, in altri termini, non si occupa di cose "interne" ad una impresa, ma degli scambi tra questa e il resto del Mondo.

Con queste premesse, possiamo distinguere a seconda del perimetro considerato:
- se il perimetro è la singola impresa, si parla di **Bilancio Civilistico**
- se il perimetro è un Gruppo di aziende connesse, si parla di **Bilancio Consolidato**

Questa distinzione è indipendente da quella precedente: un bilancio consolidato, ad esempio, può essere redatto secondo i principi OIC, o secondo i principi IAS/IFRS. Il contenuto del documento, al netto di alcune specifiche sezioni, non varierà a seconda del perimetro.

**Il nostro progetto è per il momento limitato ai _soli bilanci OIC_**.

In un futuro, si potrà espandere il progetto per includere bilanci IAS/IFRS; visto che a) sono numericamente meno numerosi b) sono più liberi nel formato
c) sono spesso in lingua inglese, affronteremo il problema quando il progetto sarà più maturo.

## Contenuto del Bilancio OIC
Il bilancio OIC è composto da diversi documenti, di finalità, origine e natura differente.

|Documento|Organo|Contenuto|Finalità|Sempre Presente|
|---------|------|---------|--------|------------|
|Prospetti|CdA|Prospetti tabellari | Rappresentazione numerica di reddito e patrimonio | Sì |
|Nota Integrativa|Cda|Testo con tabelle| Spiegazione e dettagli su quanto esposto nei prospetti | Sì|
|Relazione di Gestione|CdA|Testo con tabelle| Rappresentazione degli andamenti di mercato e dei rischi aziendali | Sì|
|Relazione del Collegio Sindacale|Collegio Sindacale|Testo|Relazione dell'Organo di Controllo Interno|Sì|
|Relazione della Società di Revisione|Società di Revisione|Testo|Revisione dell'Organo di Controllo Esterno|No|
|Verbale di approvazione|Assemblea Ordinaria|Testo|Verbale con cui l'assemblea approva il bilancio|No|

In particolare:
- la Relazione della Società di Revisione è presente nei casi in cui c'è obbligo legale di Revisione, o l'azienda sceglie volontariamente di fare revisionare il proprio Bilancio
- il Verbale di approvazione c'è per i bilanci civilistici, mentre di norma i bilanci consolidati non sono oggetto di approvazione da parte dell'Assemblea dei soci.

### Approfondimento sui Prospetti

I prospetti che compongono la parte numerica del Bilancio sono tre:

- lo **Stato Patrimoniale**, che rappresenta le attività presenti in azienda, in contrapposizione alle passività e al Patrimonio Netto, ad una determinata data; è quindi un documento di stock;
- il **Conto Economico**, che rappresenta i ricavi e i costi che l'azienda ha registrato in un periodo di tempo, e - per differenza - il risultato di esercizio; è un documento di flusso;
- il **Rendiconto Finanziario**, che rappresenta le entrate e le uscite finanziarie che l'azienda ha registrato in un determinato periodo di tempo, e - per differenza - la creazione o distruzione di cassa nel periodo; è un documento di flusso.

Alcune note:
- i prospetti hanno due gradi di libertà, non tre: dati due documenti, il terzo è ricostruibile (su base analitica, e meno precisa)anche nel caso sia assente (può succedere ad esempio, che, al di sotto di determinate dimensioni aziendali, il Rendiconto Finanziario  non sia presente);
- in generale, quando riesponiamo il Bilancio, tendiamo a ricostruire il Rendiconto Finanziario basandoci sullo Stato Patrimoniale e sul Conto Economico.
- ciò che rende differente il Conto Economico dal Rendiconto Finanziario è, in primis, il concetto di _competenza economica_, funzionale a calcolare il reddito che si è generato in un esercizio (che, dopotutto, è funzionale anche a pagarci le tasse sopra). In altri termini, se oggi pago un affitto di 100 euro, in via anticipata:
   - dal punto di vista del Rendiconto Finanziario: ho una uscita di 100 euro;
   - dal punto di vista economico: ho una uscita di 50 euro, perché gli altri 50 euro "competono" al prossimo anno, quando godrò del bene nel nuovo esercizio.

# Mission del progetto
Il bilancio è un documento estremamente pesante e complesso: ci sono bilanci che possono superare le 500 pagine.

Purtroppo, questo significa che le informazioni per noi rilevanti - il segnale - sono spesso immerse nel rumore, composto da:
- boilerplates di origine legale, senza nessuna utilità per noi. I criteri di valutazione, con cui si apre solitamente la Nota
Integrativa, sono per noi a basso valore aggiunto, _salvo casi particolari_;
- elementi che non hanno valore aggiunto per l'analisi creditizia, come ad esempio la spaccatura delle imposte pagate tra correnti,
anticipate, posticipate.

Lo scopo del progetto non è di "capire il bilancio": quello è ancora compito di un professionista che abbia la capacita di leggere
e comprendere ciò che i numeri significano in termini di realtà fattuale.

Lo scopo è cercare di separare il segnale dal rumore, in modo che questo professionista non perda tempo ad esaminare parti non rilevanti, o un inutile livello di dettaglio, con il rischio di perdere le informazioni realmente rilevanti.